In [0]:
%run ../utils/init.py


In [0]:
df_bronze = spark.read.format("delta").load(f"{BRONZE}/insee_populations/")
print(f"Lignes bronze : {df_bronze.count()}")
df_bronze.printSchema()

In [0]:
df_silver = (df_bronze
    # Supprimer communes sans population
    .filter(F.col("population_totale") > 0)
    # Indicateur : population comptée à part (différence entre municipale et totale)
    .withColumn("population_extra",
        F.col("population_totale") - F.col("population_municipale"))
    # Renommage
    .withColumnRenamed("nom_commune", "city")
    # Déduplication
    .dropDuplicates(["code_commune"])
    # Sélection finale
    .select(
        "code_commune", "city", "code_departement",
        "population_municipale", "population_totale",
        "population_extra", "annee_recensement"
    )
)

print(f"Lignes silver : {df_silver.count()}")
df_silver.show(3)

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("code_departement") \
    .save(f"{SILVER}/insee_populations/")

print("✅ Silver INSEE Populations écrit")